Get Spotify data and explore it!

In [1]:

%load_ext autoreload 
%autoreload 2

import os
import sys
from dotenv import load_dotenv
import logging

sys.path.append('./src')
from api.lyrics_client import LyricsClient
from api.spotify_client import SpotifyClient
from embeddings.embedding_manager import EmbeddingManager # need to downgrade onnxruntime 1.15.1 !!!
from models import SearchQuery
from search.hybrid_search import HybridSearchEngine
from services.lyrics_fetch_service import LyricsFetchService
from services.sync_service import SyncService
from storage.csv_tables import PlaylistsTable, SongsTable, PlaylistTracksTable, LyricsTable
from storage.state_store import StateStore

logger = logging.getLogger(__name__)
logger.setLevel('INFO')

resource module not available on Windows


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Isabella\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# Load environment variables
load_dotenv()

# get secrets
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI", "http://localhost:3000/callback")

In [3]:
emb_manager = EmbeddingManager()
search_engine = HybridSearchEngine(emb_manager)
my_sync_service = SyncService()
sp = SpotifyClient()
lc = LyricsClient()
lfs = LyricsFetchService()

In [4]:
sync_results = my_sync_service.sync_all(spotify=sp, lyrics_client=lc, lyrics_service=lfs, search_engine=search_engine)

Add of existing embedding ID: 2ETCFOsoYyz1MgIl9ZVv59
Add of existing embedding ID: 0dqGfTwHikcEdw5CSoqoK4
Add of existing embedding ID: 47NGN2PkaBo0ap4XF5LbJ2
Add of existing embedding ID: 1JP8SxPUCXv9I0Fl6xL8Ij
Add of existing embedding ID: 1aLOHf7qzU8CZaEQike0wM
Add of existing embedding ID: 1DhyA0JkflY5hjnTiEYKxd
Add of existing embedding ID: 0sOGIWu9bxNHu6xnCLg55U
Add of existing embedding ID: 5OXoNoG5b1hkwyI6bJW76C
Add of existing embedding ID: 4WPeRQDdNZoxMgu7i8cHEq
Add of existing embedding ID: 2pnlEda4uCwyl1HNJ4X2iz
Add of existing embedding ID: 4DZg30ivmW60XGmd8grz0X
Add of existing embedding ID: 0XOSOGUGvs8RlfV59Abc0t
Add of existing embedding ID: 79Ore5BQ07MSJHRjUs1K3x
Add of existing embedding ID: 1ssxK9B6clZ8Gku2bYQHTh
Add of existing embedding ID: 3WFinnHQKSEhVOZmYNR0Kd
Add of existing embedding ID: 74Me3OJwv00j8NLFkY8kMk
Add of existing embedding ID: 6Df0RNShcnfIJmrRoJ6Gc9
Add of existing embedding ID: 1mduxXxLvhu5OzebDWMYqX
Add of existing embedding ID: 1EQZbseQ6EVX8Jdh

Tokenize texts:   0%|          | 0/1517 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/1517 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1517 [00:00<?, ?it/s]

In [5]:
sync_results

{'playlists_synced': 0,
 'playlists_skipped': 36,
 'playlists_removed': 0,
 'songs_upserted': 0,
 'tracks_linked': 0,
 'links_removed': 50,
 'songs_deleted': 50,
 'lyrics_deleted': 18,
 'lyrics_job_started': False,
 'lyrics_enqueued': 0,
 'playlists_deleted': 1}

In [6]:
lfs.status()

{'running': False,
 'paused': False,
 'cancelled': False,
 'enqueued': 0,
 'processed': 0,
 'succeeded': 0,
 'failed': 0,
 'started_at': '2025-10-22T19:37:05.688149',
 'last_update': '2025-10-22T19:37:05.688149',
 'queue_size': 0,
 'settings': {'max_concurrency': 2,
  'rps': 1.0,
  'batch_size': 10,
  'max_per_run': 200}}

In [ ]:
len(lfs.lyrics._read_all())/len(lfs.songs._read_all())*100

In [ ]:
# TEMP: create embeddings for all retrieved lyrics

all_lyrics = []

rows = lfs.songs._read_all()
for r in rows:
    sid = r.get('song_id')
    lobj = lfs.lyrics.get_lyrics(sid)
    if lobj != 'lyrics unavailable':

        item = {
            'song_id': sid,
            'title': r.get('title') or '',
            'artist': r.get('artist') or '',
            'album': r.get('album') or '', 
            'duration_ms': str(r.get('duration_ms') or 0),
            'popularity': str(r.get('popularity') or 0),
            'release_date': r.get('release_date') or '', 
            'lyrics_text': lobj.get('lyrics_text') or '',
            'language': lobj.get('language') or '',
            'lyrics_source': lobj.get('source'),
            'lyrics_length': str(len(lobj.get('lyrics_text') or ''))
        }
        all_lyrics.append(item)

emb_manager.add_batch(all_lyrics)

In [37]:
search_query = SearchQuery(query='perché', search_type='keyword')
res = search_engine.search(search_query)
res.results

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[SearchResult(song=Song(song_id='0zCeKYqKQielrhrox5mh5l', original_id='0zCeKYqKQielrhrox5mh5l', title='Vieni A Vedere Perchè - Live', artist='Cesare Cremonini', album='CREMONINI LIVE: STADI 2022 + IMOLA', isrc_id='ITUM72201364', duration_ms=319000, popularity=26, release_date='2022-10-27', added_at=datetime.datetime(2025, 10, 21, 23, 26, 9, 803486), source='spotify'), lyrics_excerpt='me Ma questa non è (la verità) (Vieni a vedere perché) Grazie Mi vedono sempre ridere Ma questa non è la', relevance_score=1.0, match_type='exact_phrase', matched_terms=['perché']),
 SearchResult(song=Song(song_id='7LYUIppHHkuKeIyfWwUEvj', original_id='7LYUIppHHkuKeIyfWwUEvj', title='Vieni A Vedere Perchè', artist='Cesare Cremonini', album='Bagus', isrc_id='NLE880200004', duration_ms=253626, popularity=61, release_date='2002-11-15', added_at=datetime.datetime(2025, 10, 21, 23, 26, 9, 804582), source='spotify'), lyrics_excerpt='me Ma questa non è la verità Vieni a vedere perché Mi vedono sempre ridere Ma qu

In [ ]:
#exact search score = 1 if found, 0 otherwise
#fuzzy search score between 0 and 100
#keyword search score (tfdid 0-1, bm25 unbounded)
#semantic search score 0-1